In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine

In [4]:
def extract():
    print("[Extract] Fetching data from a Public API")
    url = "http://universities.hipolabs.com/search?country=India"
    response = requests.get(url)
    data = response.json()  #Returns a list of dictionaries (JSON)
    print(f"[Extract] Successfully retrieved {len(data)} universities in India")
    return data

[Extract] Fetching data from a Public API
[Extract] Successfully retrieved 474 universities in India
[{'web_pages': ['https://atharvacoe.ac.in/'], 'name': 'Atharva College of Engineering', 'country': 'India', 'alpha_two_code': 'IN', 'domains': ['atharvacoe.ac.in'], 'state-province': 'Mumbai'}, {'web_pages': ['http://bpitindia.ac.in/'], 'name': 'Bhagwan Parshuram Institute of Technology', 'country': 'India', 'alpha_two_code': 'IN', 'domains': ['bpitindia.edu.in', 'bpitindia.in'], 'state-province': 'Delhi'}, {'web_pages': ['https://www.upes.ac.in/'], 'name': 'University of Petroleum and Energy Studies', 'country': 'India', 'alpha_two_code': 'IN', 'domains': ['upes.ac.in'], 'state-province': 'Dehradun'}, {'web_pages': ['https://www.ashoka.edu.in/'], 'name': 'Ashoka University', 'country': 'India', 'alpha_two_code': 'IN', 'domains': ['ashoka.edu.in'], 'state-province': 'Haryana'}, {'web_pages': ['https://www.iiits.ac.in/'], 'name': 'Indian Institute of Information Technology Sri City', 'co

In [5]:
def transform(raw_data):
    print("[Transform] Cleaning and filtering data...")

    # Load raw JSON data into a DataFrame (Rows and Columns)
    df = pd.DataFrame(raw_data)

    # Filter: Keep only universities 
    df = df[df["name"].str.contains("university", case=False, na=False)]

    # Clean-up: The 'domains' and 'web_pages' columns contain Python lists.
    df["domains"] = df["domains"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    df["web_pages"] = df["web_pages"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    
    # Drop the default index to keep the table neat
    df = df.reset_index(drop=True)

    print(f"[Transform] Filtered down to {len(df)} universities.")
    return df


[Transform] Cleaning and filtering data...
[Transform] Filtered down to 291 universities.
                                  web_pages  \
0                   https://www.upes.ac.in/   
1                https://www.ashoka.edu.in/   
2                        http://www.lpu.in/   
3                  http://www.ncuindia.edu/   
4                     http://www.ddu.ac.in/   
..                                      ...   
286                    https://tsvu.nic.in/   
287               https://satavahana.ac.in/   
288              https://www.skltshu.ac.in/   
289  https://www.telanganauniversity.ac.in/   
290            https://www.chitkara.edu.in/   

                                                  name country alpha_two_code  \
0           University of Petroleum and Energy Studies   India             IN   
1                                    Ashoka University   India             IN   
2                       Lovely Professional University   India             IN   
3                    

,web_pages,name,country,alpha_two_code,domains,state-province
0,https://www.upes.ac.in/,University of Petroleum and Energy Studies,India,IN,upes.ac.in,Dehradun
1,https://www.ashoka.edu.in/,Ashoka University,India,IN,ashoka.edu.in,Haryana
2,http://www.lpu.in/,Lovely Professional University,India,IN,lpu.in,Punjab
3,http://www.ncuindia.edu/,NorthCap University,India,IN,ncuindia.edu,Haryana
4,http://www.ddu.ac.in/,Dharamsinh Desai University,India,IN,ddu.ac.in,Gujarat
...,...,...,...,...,...,...
286,https://tsvu.nic.in/,P. V. Narasimha Rao Telangana Veterinary Unive...,India,IN,tsvu.nic.in,Telangana
287,https://satavahana.ac.in/,Satavahana University,India,IN,satavahana.ac.in,Telangana
288,https://www.skltshu.ac.in/,Sri Konda Laxman Telangana State Horticultural...,India,IN,skltshu.ac.in,Telangana
289,https://www.telanganauniversity.ac.in/,Telangana University,India,IN,telanganauniversity.ac.in,Telangana


In [6]:
def load(clean_df):
    print("[Load] Connecting to database and saving data...")
    
    # Create a local SQLite database file named 'universities.db'
    engine = create_engine("sqlite:///universities.db")
    
    # Save the data into a table named 'ind_uni'
    clean_df.to_sql("ind_uni", con=engine, if_exists="replace", index=False)
    
    print("[Load] Data pipeline successfully completed! Saved to 'universities.db'.")

In [10]:
def extract():
    print("[Extract] Fetching data from a Public API")
    url = "http://universities.hipolabs.com/search?country=India"
    response = requests.get(url)
    data = response.json()  #Returns a list of dictionaries (JSON)
    print(f"[Extract] Successfully retrieved {len(data)} universities in India")
    return data
    
def transform(raw_data):
    print("[Transform] Cleaning and filtering data...")

    # Load raw JSON data into a DataFrame (Rows and Columns)
    df = pd.DataFrame(raw_data)

    # Filter: Keep only universities 
    df = df[df["name"].str.contains("university", case=False, na=False)]

    # Clean-up: The 'domains' and 'web_pages' columns contain Python lists.
    df["domains"] = df["domains"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    df["web_pages"] = df["web_pages"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    
    # Drop the default index to keep the table neat
    df = df.reset_index(drop=True)

    print(f"[Transform] Filtered down to {len(df)} universities.")
    return df

def load(clean_df):
    print("[Load] Connecting to database and saving data...")
    
    # Create a local SQLite database file named 'universities.db'
    engine = create_engine("sqlite:///universities.db")
    
    # Save the data into a table named 'ind_uni'
    clean_df.to_sql("ind_uni", con=engine, if_exists="replace", index=False)
    
    print("[Load] Data pipeline successfully completed! Saved to 'universities.db'.")
    
if __name__ == "__main__":
    # Run Extract
    raw_data = extract()
    
    # Run Transform
    transformed_data = transform(raw_data)
    
    # Run Load
    load(transformed_data)

[Extract] Fetching data from a Public API
[Extract] Successfully retrieved 474 universities in India
[Transform] Cleaning and filtering data...
[Transform] Filtered down to 291 universities.
[Load] Connecting to database and saving data...
[Load] Data pipeline successfully completed! Saved to 'universities.db'.


In [ ]:
# Read the data back from the database table you just created
engine = create_engine("sqlite:///universities.db")
verification_df = pd.read_sql("SELECT * FROM ind_uni LIMIT 5", con=engine)

# Display the first 5 rows
verification_df